In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


In [2]:
# !wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [3]:
df = pd.read_csv('../data/course_lead_scoring_2026.csv')

In [4]:
all_ = list(df.dtypes.index)

categorical = list(df.dtypes[df.dtypes=='str'].index)
numerical = list(set(all_) - set(categorical))

numerical.remove('converted')

In [5]:
numerical

['annual_income',
 'lead_score',
 'number_of_courses_viewed',
 'interaction_count']

In [6]:
# df.dtypes

In [7]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [8]:
'''
    Data preparation
    ---------------------------------
Check if the missing values are presented in the features.
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0
'''

df[numerical] = df[numerical].fillna(0.0)
df[categorical] = df[categorical].fillna('NA')
# df.isnull().sum()

# In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.

In [9]:
'''
    Question 1
What is the most frequent observation (mode) for the column industry?
'''

df.industry.mode()
# technology

0    technology
Name: industry, dtype: str

In [10]:
'''
    Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

interaction_count and lead_score => 0.915746
number_of_courses_viewed and lead_score  => 0.757204
number_of_courses_viewed and interaction_count => 0.757204
annual_income and interaction_count => 0.122842
Only consider the pairs above when answering this question.
'''
df[numerical].corr()

# interaction_count and lead_score

,annual_income,lead_score,number_of_courses_viewed,interaction_count
annual_income,1.000000,0.229496,0.161300,0.122842
lead_score,0.229496,1.000000,0.757204,0.915746
number_of_courses_viewed,0.161300,0.757204,1.000000,0.721609
interaction_count,0.122842,0.915746,0.721609,1.000000


In [11]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# resetting the indexes
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# getting our target variable and dropping it from the dataframe
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [12]:
'''
    Question 3
    --------------------------
Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

industry
location
lead_source
employment_status
'''
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.converted)

mutual_score = df_full_train[categorical].apply(mutual_info_churn_score).round(2)
mutual_score

# lead_source

lead_source          0.03
industry             0.00
employment_status    0.02
location             0.00
dtype: float64

In [13]:
'''
    Question 4
    ------------------------------
Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

0.55
0.65
0.75
0.85
'''

# one-hot encoding to get X_train and X_val
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

# model training
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# calculate model accuracy on validation dataset
y_pred_on_val = model.predict_proba(X_val)[:,1]

prediction = (y_pred_on_val >= 0.5).astype(int)
ans = (prediction == y_val).mean()
ans

# 0.645

np.float64(0.645)

In [14]:
'''
    Question 5
    ---------------------------
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

'lead_source'
'number_of_courses_viewed'
'interaction_count'

Note: The difference doesn't have to be positive.
'''

def train_me(numerical):
    # one-hot encoding to get X_train and X_val
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    # model training
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # calculate model accuracy on validation dataset
    y_pred_on_val = model.predict_proba(X_val)[:,1]
    
    prediction = (y_pred_on_val >= 0.5).astype(int)
    ans = (prediction == y_val).mean()
    return ans


numerical_original = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']

numerical_one = ['interaction_count', 'number_of_courses_viewed', 'annual_income']
numerical_two = ['interaction_count', 'lead_score', 'annual_income']
numerical_three = ['lead_score', 'number_of_courses_viewed', 'annual_income']

original_accuracy = train_me(numerical_original)
minus_lead_source = train_me(numerical_one)
minus_number_of_courses_viewed = train_me(numerical_two)
minus_interaction_count = train_me(numerical_three)

original_accuracy - minus_lead_source, original_accuracy - minus_number_of_courses_viewed, original_accuracy - minus_interaction_count

# lead_source

(np.float64(0.0010000000000000009),
 np.float64(0.0020000000000000018),
 np.float64(0.04400000000000004))

In [15]:
'''
    Question 6
    ---------------------------
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

0.000001
0.00001
0.0001
0.001
Note: If there are multiple options, select the smallest C.

'''

def train_me(c=1.0):
    # one-hot encoding to get X_train and X_val
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    
    # model training
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # calculate model accuracy on validation dataset
    y_pred_on_val = model.predict_proba(X_val)[:,1]
    
    prediction = (y_pred_on_val >= 0.5).astype(int)
    ans = (prediction == y_val).mean()
    return ans

custom_c = [0.000001, 0.00001, 0.0001, 0.001]

ans = []
for c in custom_c:
    resp = train_me(c)
    ans.append(resp)

# sorted(ans)
# ans
min(ans)

# 0.598

np.float64(0.598)